##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the Fashion MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# 2. Preprocess the images for Transfer Learning
def preprocess_fashion_mnist(images):
    images = np.expand_dims(images, axis=-1)
    images = np.repeat(images, 3, axis=-1)
    images = tf.image.resize(images, [32, 32])
    images = preprocess_input(images)
    return images.numpy()

print("Preprocessing data...")
x_train_preprocessed = preprocess_fashion_mnist(x_train)
x_test_preprocessed = preprocess_fashion_mnist(x_test)

y_train_cat = tf.keras.utils.to_categorical(y_train, 10)
y_test_cat = tf.keras.utils.to_categorical(y_test, 10)

# 3. Load the Pretrained Model (MobileNetV2)
base_model = MobileNetV2(
    input_shape=(32, 32, 3), 
    include_top=False, 
    weights='imagenet'
)

base_model.trainable = False

# 4. Build the new Classification Head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), 
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2), 
    layers.Dense(10, activation='softmax') 
])

model.summary()

# 5. Compile and Train the Model (Feature Extraction)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Training frozen model...")
history = model.fit(
    x_train_preprocessed, y_train_cat,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# 6. Fine-Tuning 
base_model.trainable = True

print(f"Number of layers in the base model: {len(base_model.layers)}")
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning the model...")
history_fine = model.fit(
    x_train_preprocessed, y_train_cat,
    epochs=5, 
    batch_size=64,
    validation_split=0.2
)

# 7. Evaluate on the Test Set
test_loss, test_acc = model.evaluate(x_test_preprocessed, y_test_cat, verbose=2)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

Preprocessing data...


C:\Users\A NWAR DAHAN\AppData\Local\Temp\ipykernel_13064\3482500497.py:27: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 10s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Training frozen model...
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 36s 42ms/step - accuracy: 0.3881 - loss: 1.6764 - val_accuracy: 0.4246 - val_loss: 1.5675
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 28s 37ms/step - accuracy: 0.4254 - loss: 1.5451 - val_accuracy: 0.4376 - val_loss: 1.5300
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 30s 41ms/step - accuracy: 0.4375 - loss: 1.5111 - val_accuracy: 0.4415 - val_loss: 1.5111
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - accuracy: 0.4431 - loss: 1.4928 - val_accuracy: 0.4448 - val_loss: 1.4993
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 29s 39ms/step - accuracy: 0.4479 - loss: 1.4784 - val_accuracy: 0.4482 - val_loss: 1.4912
Number of layers in the base model: 154
Fine-tuning the model...
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 112s 132ms/step - accuracy: 0.6075 - loss: 2.1197 - val_accuracy: 0.2183 - val_loss: 7.0690
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 97s 130ms/step - accuracy: 0.7520 - loss: 0.7234 - val_accuracy: 0.4049 - val_loss: 2.2160
Epoc